In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 26. LoRA와 가지치기 — 파라미터 효율적 파인튜닝 [CPU/GPU 벤치마크 ⑫]

> **학습 목표**
> - LoRA의 저랭크 근사 수학을 유도한다
> - 가지치기(Pruning)와 희소성(sparsity)의 의미를 안다
> - LoRA 학습이 전체 파인튜닝 대비 얼마나 빠른지 측정한다

## 26.1 PEFT의 필요성

풀 파인튜닝: 모든 파라미터 갱신 → 큰 GPU 메모리, 느림.

**PEFT** (Parameter-Efficient Fine-Tuning): 일부만 갱신.
- LoRA, Adapter, Prefix Tuning, Prompt Tuning
- 1% 미만 파라미터만으로 풀 FT와 비슷한 성능

## 26.2 LoRA — 저랭크 적응

가중치 업데이트 $\Delta W$가 저랭크(low-rank)라고 가정:
$$W' = W + \Delta W = W + B A$$

- $A \in \mathbb{R}^{r \times d}$, $B \in \mathbb{R}^{d \times r}$
- $r \ll d$ (보통 4~64)
- 파라미터: $d^2 \to 2rd$ (큰 절약)

초기화:
- $A \sim \mathcal{N}(0, \sigma^2)$ (랜덤)
- $B = 0$ (처음에 $\Delta W = 0$)

스케일링: $\Delta W = \frac{\alpha}{r} B A$

학습: $W$는 고정, $A, B$만 학습.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

class LoRALinear(nn.Module):
    """LoRA가 Application된 LinearLayer."""
    def __init__(self, in_features, out_features, r=8, alpha=16, bias=True):
        super().__init__()
        self.base = nn.Linear(in_features, out_features, bias=bias)
        # base Weight 고정
        self.base.weight.requires_grad = False
        if bias:
            self.base.bias.requires_grad = False
        # LoRA 어댑터
        self.A = nn.Parameter(torch.randn(r, in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_features, r))
        self.scaling = alpha / r

    def forward(self, x):
        # base + LoRA delta
        base_out = self.base(x)
        lora_out = (x @ self.A.t()) @ self.B.t() * self.scaling
        return base_out + lora_out

# Parameter Count Comparison
d = 4096
full = nn.Linear(d, d)
lora = LoRALinear(d, d, r=8, alpha=16)
full_params = sum(p.numel() for p in full.parameters())
lora_params = sum(p.numel() for p in lora.parameters() if p.requires_grad)
print(f"Full Linear: {full_params:,} params")
print(f"LoRA (r=8): {lora_params:,} trainable params")
print(f"Ratio: {lora_params/full_params*100:.3f}%")
print("\n=> LoRA는 0.4% 파라미터만 Training!")


## 26.3 랭크 $r$의 영향

$r$이 클수록 표현력 ↑, 메모리 ↑. 작업 복잡도에 따라 적절한 $r$ 선택.
- 단순 작업: $r = 4$
- 복잡한 파인튜닝: $r = 16, 32, 64$


In [ ]:
# 랭크별 파라미터 수와 성능 (시뮬레이션)
ranks = [1, 2, 4, 8, 16, 32, 64, 128]
d = 4096
print(f"{'r':>5} | {'LoRA params':>12} | {'% of full':>10}")
print("-" * 35)
for r in ranks:
    lora_params = 2 * r * d  # A + B
    full_params = d * d
    print(f"{r:>5} | {lora_params:>12,} | {lora_params/full_params*100:>9.3f}%")

# Visualization
fig, ax = plt.subplots(figsize=(9, 5))
params_pct = [2 * r * d / (d * d) * 100 for r in ranks]
ax.plot(ranks, params_pct, 'o-', linewidth=2, markersize=8)
ax.set_xlabel('LoRA rank r')
ax.set_ylabel('Trainable params (%)')
ax.set_title(f'LoRA: rank vs trainable params (d={d})')
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('../figures/ch26_lora_ranks.png', dpi=100, bbox_inches='tight')
plt.show()


## 26.4 어디에 LoRA 적용할까

트랜스포머에서 LoRA를 적용할 위치:
- $W_q, W_k, W_v, W_o$ (어텐션)
- FFN의 $W_1, W_2$

보통 어텐션의 Q, V에 적용 (QLoRA 등은 전부). 적용 위치가 성능에 영향.


In [ ]:
# 어텐션에 LoRA 적용
class LoRAAttention(nn.Module):
    def __init__(self, d_model, n_heads, r=8):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        # base 가중치 (고정)
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_qkv.weight.requires_grad = False
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.W_o.weight.requires_grad = False
        # LoRA 어댑터 (QKV에만)
        self.lora_qkv_A = nn.Parameter(torch.randn(r, d_model) * 0.01)
        self.lora_qkv_B = nn.Parameter(torch.zeros(3 * d_model, r))
        self.scaling = 16 / r

    def forward(self, x, mask=None):
        B, T, D = x.shape
        base = self.W_qkv(x)
        # LoRA delta
        delta = (x @ self.lora_qkv_A.t()) @ self.lora_qkv_B.t() * self.scaling
        qkv = base + delta
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores + mask
        weights = F.softmax(scores, dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out), weights

# 테스트
d, h = 64, 4
attn = LoRAAttention(d, h, r=4)
x = torch.randn(2, 10, d)
out, w = attn(x)
print(f"Output: {out.shape}")
n_trainable = sum(p.numel() for p in attn.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in attn.parameters())
print(f"Trainable: {n_trainable:,} / Total: {n_total:,} ({n_trainable/n_total*100:.2f}%)")


## 26.5 가지치기 (Pruning)

### 비구조적 가지치기
개별 가중치를 0으로:
$$\tilde{W}_{ij} = W_{ij} \cdot \mathbb{1}[|W_{ij}| > \tau]$$

문제: 희소하지만 하드웨어 가속 어려움.

### 구조적 가지치기
채널/헤드 단위로 제거:
- 적은 중요도의 헤드 제거 (attention head pruning)
- 필터/채널 제거 (CNN)
- 하드웨어 친화적

**Lottery Ticket Hypothesis**: 밀집 네트워크 안에 성능 좋은 희소 서브네트워크가 존재.


In [ ]:
# 가지치기 예시
def magnitude_prune(W, sparsity=0.5):
    """Magnitude 기반 비Structure적 Pruning."""
    threshold = torch.quantile(W.abs().flatten(), sparsity)
    mask = (W.abs() > threshold).float()
    return W * mask, mask

# Test
torch.manual_seed(0)
W = torch.randn(100, 100) * 0.1
for sparsity in [0.5, 0.7, 0.9, 0.95]:
    W_pruned, mask = magnitude_prune(W, sparsity)
    actual_sparsity = 1 - mask.mean()
    print(f"목표 sparsity {sparsity}: 실제 {actual_sparsity:.4f}, "
          f"남은 Circle소 {int(mask.sum())}개")

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
W_orig = torch.randn(50, 50) * 0.1
for ax, sp in zip(axes, [0.0, 0.7, 0.9]):
    W_p, _ = magnitude_prune(W_orig, sp)
    ax.imshow(W_p.numpy(), cmap='RdBu', vmin=-0.3, vmax=0.3)
    ax.set_title(f'Sparsity = {sp}')
plt.tight_layout()
plt.savefig('../figures/ch26_pruning.png', dpi=100, bbox_inches='tight')
plt.show()


## 26.6 [CPU/GPU 벤치마크 ⑫] Full FT vs LoRA vs QLoRA


In [ ]:
# Full FT vs LoRA 비교
from llm_math.bench import time_fn

# 가짜 모델 (큰 선형층)
class TinyModel(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.fc1 = nn.Linear(d, d * 4)
        self.fc2 = nn.Linear(d * 4, d)
        self.fc3 = nn.Linear(d, d)
    def forward(self, x):
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))

# Full FT
def make_full_ft():
    model = TinyModel()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    return model, opt

# LoRA
class TinyModelLoRA(nn.Module):
    def __init__(self, d=512, r=8):
        super().__init__()
        self.fc1 = nn.Linear(d, d * 4)
        self.fc2 = nn.Linear(d * 4, d)
        self.fc3 = nn.Linear(d, d)
        # base 고정
        for p in self.parameters():
            p.requires_grad = False
        # LoRA 어댑터 (base fc1/fc2/fc3를 LoRA 버전으로 교체하지 않고
        # base 위에 delta만 더하는 방식)
        self.lora_A1 = nn.Parameter(torch.randn(r, d) * 0.01)
        self.lora_B1 = nn.Parameter(torch.zeros(d * 4, r))
        self.lora_A2 = nn.Parameter(torch.randn(r, d * 4) * 0.01)
        self.lora_B2 = nn.Parameter(torch.zeros(d, r))
        self.lora_A3 = nn.Parameter(torch.randn(r, d) * 0.01)
        self.lora_B3 = nn.Parameter(torch.zeros(d, r))
        self.scaling = 16 / r
    def forward(self, x):
        # fc1 + LoRA delta
        h1 = F.relu(self.fc1(x) + (x @ self.lora_A1.t()) @ self.lora_B1.t() * self.scaling)
        # fc2 + LoRA delta
        h2 = F.relu(self.fc2(h1) + (h1 @ self.lora_A2.t()) @ self.lora_B2.t() * self.scaling)
        # fc3 + LoRA delta
        out = self.fc3(h2) + (h2 @ self.lora_A3.t()) @ self.lora_B3.t() * self.scaling
        return out

def make_lora():
    model = TinyModelLoRA(r=8)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=1e-4)
    return model, opt

# 학습 한 스텝 시간 비교
def bench_step(model, opt, x, y, loss_fn, device='cpu'):
    def step():
        opt.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        opt.step()
        return loss
    return step

d = 256
loss_fn = nn.MSELoss()
x = torch.randn(8, d)
y = torch.randn(8, d)

# Model을 d=256으로 Generation
def make_full_ft(d=256):
    model = TinyModel(d=d)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    return model, opt

def make_lora(d=256):
    model = TinyModelLoRA(d=d, r=8)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=1e-4)
    return model, opt

model_full, opt_full = make_full_ft(d=d)
model_lora, opt_lora = make_lora(d=d)

n_full = sum(p.numel() for p in model_full.parameters() if p.requires_grad)
n_lora = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
print(f"Full FT: {n_full:,} trainable params")
print(f"LoRA:    {n_lora:,} trainable params ({n_lora/n_full*100:.2f}%)")

res_full = time_fn(bench_step(model_full, opt_full, x, y, loss_fn), device='cpu', warmup=2, repeat=5)
res_lora = time_fn(bench_step(model_lora, opt_lora, x, y, loss_fn), device='cpu', warmup=2, repeat=5)
print(f"\nFull FT 1 step: {res_full['mean_ms']:.3f} ms")
print(f"LoRA 1 step:    {res_lora['mean_ms']:.3f} ms")
print(f"Speed Improvement: {res_full['mean_ms'] / res_lora['mean_ms']:.2f}x")
print("\n=> LoRA는 Training 파라미터가 적어 Memory와 Speed 모두 유리.")


## 26.7 핵심 정리

| 기법 | 학습 파라미터 | 메모리 | 성능 |
|---|---|---|---|
| Full FT | 100% | 큼 | 최고 |
| LoRA | ~1% | 작음 | 비슷 |
| QLoRA | ~1% + 4-bit base | 매우 작음 | 약간 손실 |
| Adapter | ~5% | 보통 | 비슷 |
| Prefix Tuning | ~0.1% | 매우 작음 | 약간 손실 |

## 연습문제

1. LoRA rank $r = 1, 4, 16, 64$로 학습하고 성능을 비교하라.
2. LoRA를 QKV 전부에 적용한 것과 Q에만 적용한 것을 비교하라.
3. 50%, 70%, 90% 가지치기한 모델의 정확도를 비교하라.
4. Full FT vs LoRA의 학습 시간과 메모리를 측정하라.
5. LoRA가 저랭크 가정이 의미 있는 이유를 설명하라.

> 해설: `solutions/ch26_solutions.ipynb`
